In [1]:
import numpy as np

from dinosaw.helpers import get_model, get_features
from dinosaw.utils import do_2D_pca, seed_everything
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

SEED = 100001
DEVICE = 'cuda:0'

seed_everything(SEED)

/home/ronan/Documents/phd/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
selected_model = 'dv2'
model = get_model(selected_model, '../../trained_models', device=DEVICE, conf_path='../../dinov3')
n_dims = 768 if '_b' in selected_model else 384
# n_dims = 256

In [3]:
ds_folder = '../paper_figures/data/linear_probe'
micro_img = Image.open(f'{ds_folder}/homog_micros/ni_superalloy.png').convert('RGB')

noise_arr = np.random.randint(0, 256, (518, 518, 3), dtype=np.uint8)
noise_img = Image.fromarray(noise_arr).convert('RGB')

texture_img = Image.open(f'{ds_folder}/texture_ds/crosshatched_0069.jpg')


micro_feats = get_features(model, micro_img, device=DEVICE, channel_last=True)
noise_feats = get_features(model, noise_img, device=DEVICE, channel_last=True)
texture_feats = get_features(model, texture_img, device=DEVICE, channel_last=True)

In [4]:
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True

mh, mw, _ = micro_feats.shape
nh, nw, _ = noise_feats.shape
th, tw, _ = texture_feats.shape
micro_mask = gen_sample_mask((mh, mw), 'ud', STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
noise_mask = gen_sample_mask((nh, nw), 'lr', STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
texture_mask = gen_sample_mask((th, tw), 'lr+ud', STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)

micro_ramp = get_ramp('ud', mh, mw)
noise_ramp = get_ramp('lr', nh, nw)
texture_ramp = get_ramp('lr+ud', th, tw)

micro_probe = do_linear_probe(micro_feats, 'ud', probe_by_channel=False, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
noise_probe = do_linear_probe(noise_feats, 'lr', probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
texture_probe = do_linear_probe(texture_feats, 'lr+ud', probe_by_channel=False, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)

In [5]:
micro_pca = do_2D_pca(micro_feats.transpose((2, 0, 1)), post_norm='minmax')
texture_pca = do_2D_pca(texture_feats.transpose((2, 0, 1)), post_norm='minmax')

# plt.imshow(micro_pca)

In [6]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none')
    ax.add_collection(pc)

def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

def xy_to_rb_rgb(xy_map: np.ndarray) -> np.ndarray:
    if xy_map.ndim != 3 or xy_map.shape[0] != 2:
        raise ValueError("xy_map must have shape (2, H, W)")
    x = xy_map[0].clip(0.0, 1.0)
    y = xy_map[1].clip(0.0, 1.0)
    g = np.zeros_like(x)
    return np.stack([x, g, y], axis=-1)

In [10]:
plt.style.use("thesis.mplstyle")
W, H = 7, 2.3 * (3/2)
N_ROWS = 3
N_COLS = 5
fig,axs = plt.subplots(N_ROWS, N_COLS, figsize=(W, H))

axs[0, 0].imshow(micro_img)
axs[1, 0].imshow(noise_img)
axs[2, 0].imshow(texture_img)


axs[0, 0].set_title("Datasets")
axs[0, 0].set_ylabel("Micrographs")
axs[1, 0].set_ylabel("Noise")
axs[2, 0].set_ylabel("Textures")


axs[0, 1].imshow(micro_pca)
add_red_square_overlay(axs[0, 1], micro_mask, sf_h=1, sf_w=1)
axs[1, 1].imshow(noise_feats[:, :, 359])
add_red_square_overlay(axs[1, 1], noise_mask, sf_h=1, sf_w=1)

axs[2, 1].imshow(texture_pca)
add_red_square_overlay(axs[2, 1], texture_mask, sf_h=1, sf_w=1)

axs[0, 1].set_title("Training data")
axs[0, 1].set_ylabel("Full features")
axs[1, 1].set_ylabel("Specific channels")
axs[2, 1].set_ylabel("Full features")


for ax in (axs[:, 2]):
    hide_axes(ax)
    ax.set_frame_on(False)


axs[0, 2].text(0.4, 0.28, "Fit linear probe \n on sparse subset", transform=axs[0, 2].transAxes, color='red', ha='center', va='center')

for j in range(N_ROWS):
    axs[j, 2].arrow(0.0, 0.5, 0.8, 0.0, transform=axs[j, 2].transAxes, color='red', head_width=0.05, head_length=0.05)
# axs[0, 2].arrow(0.0, 0.5, 0.8, 0, transform=axs[0, 2].transAxes, color='red', head_width=0.05, head_length=0.05)
# axs[1, 2].arrow(0.0, 0.5, 0.8, 0, transform=axs[1, 2].transAxes, color='red', head_width=0.05, head_length=0.05)

axs[0, 3].imshow(micro_ramp)
add_red_square_overlay(axs[0, 3], micro_mask, sf_h=1, sf_w=1)
axs[0, 3].set_title("Target ramps")
axs[0, 3].set_ylabel("Up-down")
axs[1, 3].imshow(noise_ramp)
add_red_square_overlay(axs[1, 3], noise_mask, sf_h=1, sf_w=1)
axs[1, 3].set_ylabel("Left-right")
axs[2, 3].imshow(xy_to_rb_rgb(texture_ramp.transpose((2, 0, 1))))
axs[2, 3].set_ylabel(r"$(x, y)$ grid")
add_red_square_overlay(axs[2, 3], texture_mask, sf_h=1, sf_w=1)

axs[0, 4].imshow(micro_probe["stack_pred"], vmin=0, vmax=1)
axs[0, 4].set_title("Predictions")
axs[0, 4].set_ylabel(rf"$R^{2}: {micro_probe['stack_r_squared']:.2f}$")
axs[1, 4].imshow(noise_probe["per_channel_preds"][359], vmin=0, vmax=1)
axs[1, 4].set_ylabel(rf"$R^{2}: {noise_probe['per_channel_scores'][359]:.2f}$")

print(texture_probe["stack_pred"].shape)
texture_pred = texture_probe["stack_pred"]
texture_rgb = xy_to_rb_rgb(texture_pred.transpose((2, 0, 1)))
axs[2, 4].imshow(texture_rgb)
axs[2, 4].set_ylabel(rf"$R^{2}: {texture_probe['stack_r_squared']:.2f}$")


for ax in axs.flatten():
    hide_axes(ax)

plt.savefig("out/linear_probe_explained.pdf", bbox_inches="tight")
plt.close()

(36, 36, 2)
